# French BESS aFRR and mFRR: offline working example

This notebook reproduces the two Feature 011 reference studies from governed offline artifacts. It is a research audit, not observed battery dispatch and not a forecast, bid, optimization, or profitability guarantee. The RTE-derived files are private-repository data for authorized collaborators. Do not publish the executed notebook, screenshots, HTML exports, figures, or public CI artifacts. The source notebook is committed with empty outputs; remove or reassess the restricted fixtures before a visibility change.


## 1. The products in plain language

**Automatic Frequency Restoration Reserve (aFRR)** helps bring the power system back toward balance automatically after the fastest frequency response has acted. RTE's public values are 15-minute summaries, not the faster signal an individual asset would follow.

**Manual Frequency Restoration Reserve (mFRR)** is activated as a more deliberate balancing action. **Scheduled activation (SA)** represents a planned quarter-hour activation in the balancing process. **Direct activation (DA)** can be called directly when needed. The selected public record has source-null DA, so no direct event is fabricated.


## 2. What the source proves—and what it does not

RTE Balancing Energy 5.2 publishes French control-area aggregates, not commands sent to an individual battery. The reference scenarios apply one fixed 0.001 allocation and label every result `derived_simulation`. The aFRR inputs remain 15-minute interval averages with no interpolation or upsampling. The mFRR event shape is an explicit rectangular simulation assumption.

The audit keeps source field / series fingerprint / unit / resolution visible. For aFRR it separates requested need / activated for France / activated in France / weighted-average price. For mFRR it separates SA requested / SA satisfied / SA activated / DA status / clearing price. Prices are labelled `scenario_proxy`, not an asset invoice. The degradation value is a throughput comparison proxy, not an electrochemical aging model.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").is_file() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("run the notebook from within the Open Energy Lab checkout")
FIXTURE_DIR = REPO_ROOT / "tests/fixtures/restoration"
sns.set_theme(style="whitegrid", context="notebook")
generated_figures = {}


In [ ]:
from examples.restoration_services.generate_reference_results import (
    AFRR_SCENARIO_FILENAME,
    EXPECTED_RESULTS_FILENAME,
    MFRR_SCENARIO_FILENAME,
    build_reference_case,
    build_reference_display_payload,
    build_reference_results_payload,
    load_reference_scenario,
)

afrr_scenario = load_reference_scenario(FIXTURE_DIR / "france_2026-08-20_afrr_scenario_v1.json", fixture_directory=FIXTURE_DIR)
mfrr_scenario = load_reference_scenario(FIXTURE_DIR / "france_2026-08-15_mfrr_scenario_v1.json", fixture_directory=FIXTURE_DIR)
product_cases = {
    "AFRR": build_reference_case(afrr_scenario),
    "MFRR": build_reference_case(mfrr_scenario),
}
rebuilt_results = build_reference_results_payload(FIXTURE_DIR)
recorded_results = json.loads((FIXTURE_DIR / "france_restoration_expected_results_v1.json").read_text(encoding="utf-8"))
assert rebuilt_results == recorded_results
display_payload = build_reference_display_payload(FIXTURE_DIR)
result_set_fingerprint = rebuilt_results["result_set_fingerprint"]
notebook_display_fingerprint = display_payload["notebook_display_fingerprint"]
print(f"result_set_fingerprint={result_set_fingerprint}")
print(f"notebook_display_fingerprint={notebook_display_fingerprint}")


In [ ]:
provenance_table = pd.DataFrame(display_payload["provenance_rows"])
display(provenance_table)


In [ ]:
afrr_market_table = pd.DataFrame(display_payload["afrr_market_rows"])
mfrr_market_table = pd.DataFrame(display_payload["mfrr_market_rows"])
display(afrr_market_table)
display(mfrr_market_table)


In [ ]:
afrr_plot = afrr_market_table.copy()
afrr_plot["start_utc"] = pd.to_datetime(afrr_plot["start_utc"], utc=True)
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for series_id, group in afrr_plot[afrr_plot["unit"] == "MW"].groupby("series_id"):
    axes[0].plot(group["start_utc"], group["value"], marker="o", label=series_id)
axes[0].set_ylabel("French area volume (MW)")
axes[0].set_title("Observed RTE aFRR area aggregates")
axes[0].legend(fontsize=8, ncol=2)
for series_id, group in afrr_plot[afrr_plot["unit"] == "EUR/MWh"].groupby("series_id"):
    axes[1].plot(group["start_utc"], group["value"], marker="o", label=series_id)
axes[1].set_ylabel("Area price (EUR/MWh)")
axes[1].set_xlabel("UTC interval start")
axes[1].legend(fontsize=8)
fig.tight_layout()
generated_figures["afrr_market"] = fig
plt.show()


In [ ]:
mfrr_plot = mfrr_market_table.copy()
mfrr_plot["start_utc"] = pd.to_datetime(mfrr_plot["start_utc"], utc=True)
selected_mfrr_ids = ["mfrr_sa_requested_volume_downward", "mfrr_sa_satisfied_need_downward", "mfrr_sa_activated_volume_downward", "mfrr_da_requested_volume_downward"]
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for series_id, group in mfrr_plot[mfrr_plot["series_id"].isin(selected_mfrr_ids)].groupby("series_id"):
    axes[0].plot(group["start_utc"], group["value"], marker="o", label=series_id)
axes[0].set_ylabel("French area volume (MW)")
axes[0].set_title("Observed mFRR context; missing DA stays missing")
axes[0].legend(fontsize=8)
price_rows = mfrr_plot[mfrr_plot["series_id"] == "mfrr_clearing_price_downward"]
axes[1].plot(price_rows["start_utc"], price_rows["value"], marker="o", label="downward clearing price")
axes[1].set_ylabel("Area price (EUR/MWh)")
axes[1].set_xlabel("UTC interval start")
axes[1].legend(fontsize=8)
fig.tight_layout()
generated_figures["mfrr_market"] = fig
plt.show()


## 3. Mapping, commitments, and baseline

The source-to-simulation equation is `requested_bess_kw = requested_area_mw * 1000 * 0.001`. The same fraction applies to every interval and no request is clipped. Commitments are fixed study inputs, not bids. The controller accepts a zero baseline without activation lookahead and reserves directional capacity before replay.

The table exposes upward commitment / downward commitment / accepted baseline / reserved headroom. The aFRR commitment is 450 kW in each direction; the mFRR commitment is 400 kW in each direction.


In [ ]:
commitment_table = pd.DataFrame(display_payload["commitment_rows"])
commitment_table["reserved_power_headroom_kw"] = commitment_table["committed_capacity_kw"]
activation_table = pd.DataFrame(display_payload["activation_rows"])
display(commitment_table)
display(activation_table)


In [ ]:
physical_table = pd.DataFrame(display_payload["physical_rows"])
power_plot = physical_table.copy()
power_plot["start_utc"] = pd.to_datetime(power_plot["start_utc"], utc=True)
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)
for axis, product in zip(axes, ["AFRR", "MFRR"], strict=True):
    group = power_plot[power_plot["product"] == product]
    axis.plot(group["start_utc"], group["requested_balancing_power_kw"], marker="o", label="requested balancing")
    axis.plot(group["start_utc"], group["delivered_balancing_power_kw"], marker="x", label="delivered balancing")
    axis.plot(group["start_utc"], group["accepted_baseline_power_kw"], linestyle="--", label="accepted baseline")
    axis.set_title(f"{product}: derived requested and delivered power")
    axis.set_ylabel("Battery power (kW; charge positive)")
    axis.legend(fontsize=8)
axes[-1].set_xlabel("UTC interval start")
fig.tight_layout()
generated_figures["requested_delivered_power"] = fig
plt.show()


In [ ]:
performance_table = pd.DataFrame(display_payload["performance_rows"])
display(physical_table)
display(performance_table)
# requested balancing power / delivered balancing power / combined power
# tracking error / response delay / achieved ramp / non-performance / causes


In [ ]:
headroom_plot = physical_table.copy()
headroom_plot["start_utc"] = pd.to_datetime(headroom_plot["start_utc"], utc=True)
fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=False)
for product, group in headroom_plot.groupby("product"):
    axes[0].plot(group["start_utc"], group["soc"], marker="o", label=product)
    axes[1].plot(group["start_utc"], group["upward_power_headroom_kw"], marker="o", label=f"{product} upward")
    axes[1].plot(group["start_utc"], group["downward_power_headroom_kw"], marker="x", label=f"{product} downward")
    axes[2].plot(group["start_utc"], group["upward_energy_headroom_kwh"], marker="o", label=f"{product} upward")
    axes[2].plot(group["start_utc"], group["downward_energy_headroom_kwh"], marker="x", label=f"{product} downward")
axes[0].set_ylabel("SOC")
axes[1].set_ylabel("Power headroom (kW)")
axes[2].set_ylabel("Energy headroom (kWh)")
axes[2].set_xlabel("UTC interval start")
for axis in axes: axis.legend(fontsize=8)
fig.suptitle("SOC / upward power headroom / downward power headroom / energy headroom")
fig.tight_layout()
generated_figures["soc_headroom"] = fig
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
tracking_summary = physical_table.groupby("product", as_index=False)[["absolute_tracking_error_kw", "non_performance_duration_seconds"]].sum()
sns.barplot(data=tracking_summary, x="product", y="absolute_tracking_error_kw", ax=axes[0])
sns.barplot(data=tracking_summary, x="product", y="non_performance_duration_seconds", ax=axes[1])
axes[0].set_title("Absolute tracking error")
axes[1].set_title("Non-performance duration")
fig.tight_layout()
generated_figures["tracking_performance"] = fig
plt.show()


In [ ]:
energy_table = pd.DataFrame(display_payload["energy_rows"])
display(energy_table)
# charge / discharge / net terminal energy / throughput


In [ ]:
energy_long = energy_table.melt(id_vars="product", value_vars=["charge_energy_kwh", "discharge_energy_kwh", "net_terminal_energy_kwh", "throughput_kwh"], var_name="metric", value_name="kWh")
fig, axis = plt.subplots(figsize=(11, 5))
sns.barplot(data=energy_long, x="metric", y="kWh", hue="product", ax=axis)
axis.set_title("Battery-terminal energy and throughput")
axis.tick_params(axis="x", rotation=20)
fig.tight_layout()
generated_figures["energy_throughput"] = fig
plt.show()


In [ ]:
settlement_table = pd.DataFrame(display_payload["settlement_rows"])
display(settlement_table)
# day-ahead / capacity / activated energy / penalty / degradation / gross / net


In [ ]:
settlement_plot = settlement_table.copy()
settlement_plot["capacity_revenue_eur"] = settlement_plot["upward_capacity_revenue_eur"] + settlement_plot["downward_capacity_revenue_eur"]
settlement_plot["activation_revenue_eur"] = settlement_plot["upward_activation_revenue_eur"] + settlement_plot["downward_activation_revenue_eur"]
settlement_plot["penalty_cost_eur"] = -settlement_plot["non_performance_penalty_eur"]
settlement_plot["degradation_cost_signed_eur"] = -settlement_plot["degradation_cost_eur"]
settlement_long = settlement_plot.melt(id_vars="product", value_vars=["day_ahead_energy_revenue_eur", "capacity_revenue_eur", "activation_revenue_eur", "penalty_cost_eur", "degradation_cost_signed_eur", "gross_revenue_eur", "net_result_eur"], var_name="component", value_name="EUR")
fig, axis = plt.subplots(figsize=(12, 5))
sns.barplot(data=settlement_long, x="component", y="EUR", hue="product", ax=axis)
axis.set_title("Scenario-proxy settlement components")
axis.tick_params(axis="x", rotation=25)
fig.tight_layout()
generated_figures["settlement"] = fig
plt.show()


In [ ]:
reproducibility_table = pd.DataFrame(display_payload["reproducibility_rows"])
assert notebook_display_fingerprint == rebuilt_results["notebook_display_fingerprint"]
display(reproducibility_table)
# artifact / mapping / commitment / activation / replay / settlement / display fingerprints


In [ ]:
EXPORT_FIGURES = False
FIGURE_OUTPUT_DIRECTORY = REPO_ROOT / "tmp/restoration-services-notebook/figures"
if EXPORT_FIGURES:
    FIGURE_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
    for figure_name, figure in generated_figures.items():
        figure.savefig(FIGURE_OUTPUT_DIRECTORY / f"{figure_name}.png", dpi=150, bbox_inches="tight")
else:
    print("Figure export disabled; committed notebook state stays output-free.")


## 4. Optional live source check

The next cell is disabled by default and accepts a caller-selected UTC half-open period. It demonstrates an in-memory RTE Balancing Energy 5.2 query for `standard_afrr_data` or `standard_mfrr_data`, including short-lived OAuth. Raw responses remain in memory only and no raw response is written. Live availability is not required for offline results.

The optional ENTSO-E branch does not invent a restoration profile. It can inspect only an artifact produced by the existing Feature 010 `fr-day-ahead-symmetric-fcr-v1` lifecycle, require a time-compatible A44 `day_ahead_price`, and keep it selected by series fingerprint. It is not an automatic fallback. If the exact profile/path/period is absent, it reports that no compatible profile was selected. Use the existing Feature 010 fetch/refresh command outside this notebook to produce that normalized artifact.


In [ ]:
RUN_LIVE_QUERY = False
RUN_LIVE_ENTSOE_QUERY = False
LIVE_PERSIST_NORMALIZED_ARTIFACT = False
LIVE_QUERY_PARAMETERS = {
    "resource": "standard_afrr_data",
    "requested_start_utc": "2026-08-20T00:00:00Z",
    "requested_end_utc": "2026-08-20T01:00:00Z",
}
LIVE_ENV_FILE = REPO_ROOT / ".env"
LIVE_ENTSOE_ARTIFACT_PATH = None
live_rte_table = pd.DataFrame()
live_entsoe_selection = "no compatible profile was selected"
raw_responses = ()

if RUN_LIVE_QUERY:
    from hashlib import sha256
    from examples.restoration_services.fetch_rte_period import _rte_credentials
    from open_energy_lab.market.artifact import load_market_data_artifact
    from open_energy_lab.market.restoration_artifact import canonical_restoration_json_bytes
    from open_energy_lab.market.rte import (
        RTE_PROFILES,
        build_rte_restoration_request,
        execute_rte_request,
        parse_rte_restoration_responses,
    )

    if LIVE_PERSIST_NORMALIZED_ARTIFACT:
        raise ValueError("the notebook live check does not persist artifacts; use the explicit fetch CLI")
    profile = RTE_PROFILES[LIVE_QUERY_PARAMETERS["resource"]]
    live_start = pd.Timestamp(LIVE_QUERY_PARAMETERS["requested_start_utc"]).to_pydatetime()
    live_end = pd.Timestamp(LIVE_QUERY_PARAMETERS["requested_end_utc"]).to_pydatetime()
    request = build_rte_restoration_request(
        resource=profile.resource,
        requested_start_utc=live_start,
        requested_end_utc=live_end,
        requested_series_ids=profile.series_ids,
        required_series_ids=profile.required_series_ids,
        normalization_version="rte-balancing-energy-v5.2-notebook-live-v1",
        raw_retention_policy="DIGEST_ONLY",
        mode="FETCH",
    )
    client_id = ""
    client_secret = ""
    try:
        client_id, client_secret = _rte_credentials(LIVE_ENV_FILE)
        raw_responses = execute_rte_request(request, client_id=client_id, client_secret=client_secret)
        parsed = parse_rte_restoration_responses(request, raw_responses)
        live_rows = []
        for series in parsed.series:
            for observation in series.observations:
                live_rows.append({
                    "series_id": series.series_id,
                    "source_field": series.source_field,
                    "direction": series.direction,
                    "unit": series.canonical_unit,
                    "resolution": series.canonical_resolution,
                    "start_utc": observation.interval.start_utc,
                    "value_state": observation.value_state.value,
                    "value": observation.value,
                })
        live_rte_table = pd.DataFrame(live_rows)
        display(live_rte_table)

        if RUN_LIVE_ENTSOE_QUERY and LIVE_ENTSOE_ARTIFACT_PATH is not None:
            entsoe_artifact = load_market_data_artifact(Path(LIVE_ENTSOE_ARTIFACT_PATH))
            compatible = (
                entsoe_artifact.profile_id == "fr-day-ahead-symmetric-fcr-v1"
                and entsoe_artifact.actual_covered_period.start_utc <= live_start
                and entsoe_artifact.actual_covered_period.end_utc >= live_end
                and "day_ahead_price" in entsoe_artifact.series
            )
            if compatible:
                day_ahead_series = entsoe_artifact.series["day_ahead_price"]
                series_fingerprint = sha256(canonical_restoration_json_bytes(day_ahead_series.to_dict())).hexdigest()
                live_entsoe_selection = {
                    "profile_id": entsoe_artifact.profile_id,
                    "artifact_fingerprint": entsoe_artifact.artifact_fingerprint,
                    "series_id": "day_ahead_price",
                    "series_fingerprint": series_fingerprint,
                }
            else:
                live_entsoe_selection = "no compatible profile was selected"
        print(live_entsoe_selection)
    finally:
        client_id = ""
        client_secret = ""
        raw_responses = ()
else:
    print("Live query disabled; set RUN_LIVE_QUERY=True only for an explicit source check.")


## 5. What we can conclude

The example proves that the library can preserve exact RTE source semantics, build separate aFRR and mFRR inputs, reserve fixed directional capacity, apply BESS power/SOC/efficiency/service limits, audit delivery and headroom, and reconcile an explicit scenario-proxy settlement deterministically.

It does not prove that the public area aggregate was sent to this battery, that PT15M aFRR captures operational tracking, that the rectangular SA event was the actual order shape, that a DA event occurred, or that the proxy revenues would be realized. Operational control signals, asset contracts, bid selection, imbalance rules, prequalification, and forecast risk require additional sources and later features.
